In [1]:
from embedder import Embedder

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(len(v))
print(v[0])


384
-0.020582034180917443


In [2]:
from embedder import Embedder

embedder = Embedder()

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
print(len(documents))


72


In [5]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [6]:
query = "How does approximate nearest neighbor search work?"

query_vector = embedder.encode(query)

In [7]:
print(query_vector.shape)

(384,)


In [8]:
for doc in documents:
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        target_doc = doc
        break

In [9]:
target_doc["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [10]:
doc_vector = embedder.encode(target_doc["content"])

In [11]:
import numpy as np

similarity = np.dot(query_vector, doc_vector)

print(similarity)

0.36107008472347096


In [12]:
doc_vector = embedder.encode(target_doc["content"])

In [13]:
import numpy as np

similarity = np.dot(query_vector, doc_vector)

print(similarity)

0.36107008472347096


In [14]:
import numpy as np

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

doc = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

doc_v = embedder.encode(doc["content"])

similarity = np.dot(v, doc_v)

print(similarity)

0.36107008472347096


In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

len(chunks)

295

In [16]:
chunk_vectors = embedder.encode_batch(
    [chunk["content"] for chunk in chunks]
)

In [17]:
type(chunk_vectors)
chunk_vectors.shape

(295, 384)

In [18]:
X = chunk_vectors

In [19]:
query = "How does approximate nearest neighbor search work?"

v = embedder.encode(query)

In [20]:
scores = X.dot(v)

In [21]:
scores.shape

(295,)

In [22]:
best = scores.argmax()

print(best)

94


In [23]:
chunks[best]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [24]:
from minsearch import VectorSearch

vindex = VectorSearch(
    keyword_fields=["filename"]
)

vindex.fit(
    vectors=X,
    payload=chunks
)

In [25]:
import inspect
from minsearch import VectorSearch

print(inspect.signature(VectorSearch))

(keyword_fields=None, numeric_fields=None, date_fields=None)


In [26]:
query = "What metric do we use to evaluate a search engine?"
q = embedder.encode(query)

results = vindex.search(
    query_vector=q,
    num_results=5
)

print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


In [27]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [28]:
query = "How do I store vectors in PostgreSQL?"

text_results = text_index.search(
    query,
    num_results=5
)

text_files = [r["filename"] for r in text_results]
text_files

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [29]:
q = embedder.encode(query)

vector_results = vindex.search(
    query_vector=q,
    num_results=5
)

vector_files = [r["filename"] for r in vector_results]
vector_files

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [30]:
set(vector_files) - set(text_files)

{'02-vector-search/lessons/08-pgvector.md'}

In [31]:
query = "How do I give the model access to tools?"

# vector search
q = embedder.encode(query)

vector_results = vindex.search(
    query_vector=q,
    num_results=5
)

# text search
text_results = text_index.search(
    query,
    num_results=5
)

In [32]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [33]:
results = rrf([vector_results, text_results])

print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md


In [34]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

from openai import OpenAI
from pydantic import BaseModel

from gitsource import GithubRepositoryDataReader
from evaluation_utils import *

In [35]:
class Questions(BaseModel):
    questions: list[str]

In [36]:
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

In [37]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [38]:
for doc in documents[:3]:
    print(doc["filename"])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md


In [39]:
input_tokens = []
ground_truth = []

for doc in documents[:3]:
    user_prompt = json.dumps(
        {
            "filename": doc["filename"],
            "content": doc["content"]
        }
    )

    questions, usage = llm_structured(
        client=client,
        model="gpt-5.4-mini",
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions
    )

    input_tokens.append(usage.input_tokens)

    for question in questions.questions:
        ground_truth.append(
            {
                "question": question,
                "filename": doc["filename"]
            }
        )

    print(doc["filename"])
    print("Input tokens:", usage.input_tokens)
    print()

01-agentic-rag/lessons/01-intro.md
Input tokens: 1021

01-agentic-rag/lessons/02-environment.md
Input tokens: 1287

01-agentic-rag/lessons/03-rag.md
Input tokens: 1754



In [40]:
average_input_tokens = sum(input_tokens) / len(input_tokens)

print("Token counts:", input_tokens)
print("Average:", average_input_tokens)

Token counts: [1021, 1287, 1754]
Average: 1354.0


In [42]:
import pandas as pd

ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")

len(ground_truth)

360

In [43]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

len(chunks)

295

In [44]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [45]:
def text_search(query, num_results=5):
    return text_index.search(
        query,
        num_results=num_results
    )

In [46]:
q = ground_truth[0]["question"]

print(q)

results = text_search(q)

print(results[0]["filename"])

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
01-agentic-rag/lessons/03-rag.md


In [47]:
texts = [chunk["content"] for chunk in chunks]

X = embedder.encode_batch(texts)

In [48]:
print(X.shape)

(295, 384)


In [49]:
from minsearch import VectorSearch

vector_index = VectorSearch(
    keyword_fields=["filename"]
)

vector_index.fit(
    vectors=X,
    payload=chunks
)

In [50]:
def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)

    return vector_index.search(
        query_vector=query_vector,
        num_results=num_results
    )

In [51]:
q = ground_truth[0]["question"]

results = vector_search(q)

print(q)
print(results[0]["filename"])

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
01-agentic-rag/lessons/01-intro.md


In [52]:
def compute_relevance(record, search_function):
    results = search_function(record["question"])

    return [
        result["filename"] == record["filename"]
        for result in results
    ]

In [53]:
def hit_rate(relevance_total):
    cnt = 0

    for relevance in relevance_total:
        if True in relevance:
            cnt += 1

    return cnt / len(relevance_total)

In [54]:
def mrr(relevance_total):
    total_score = 0.0

    for relevance in relevance_total:
        for rank, rel in enumerate(relevance):
            if rel:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance_total)

In [55]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for record in ground_truth:
        relevance = compute_relevance(record, search_function)
        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [56]:
text_metrics = evaluate(ground_truth, text_search)

print(text_metrics)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}


In [57]:
vector_metrics = evaluate(
    ground_truth,
    vector_search
)

print(vector_metrics)

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}


In [58]:
vector_metrics["mrr"]

0.5486111111111112

In [59]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [60]:
ks = [1, 50, 100, 200]

results = {}

for k in ks:
    metrics = evaluate(
        ground_truth,
        lambda query: hybrid_search(query, k=k)
    )

    results[k] = metrics
    print(f"k={k}: {metrics}")

k=1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}
k=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
k=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


In [61]:
for k, metrics in results.items():
    print(k, metrics["mrr"])

1 0.6481944444444449
50 0.637916666666667
100 0.637916666666667
200 0.637916666666667
